In [0]:
df_bronze = spark.read.table("credito")

display(df_bronze)
df_bronze.printSchema()

In [0]:
from pyspark.sql.functions import col, current_timestamp, round, regexp_extract, when, expr, coalesce, lit, regexp_replace

TAXA_USD_BRL = 5.50

df_bronze = spark.read.table("credito")

df_silver_credito = df_bronze \
    .filter(col("salario_anual").isNotNull()) \
    .withColumn("salario_min_usd", coalesce(expr("try_cast(regexp_extract(salario_anual, '(\\\\d+)', 1) as double)"), lit(0.0)) * 1000) \
    .withColumn("salario_max_usd", coalesce(expr("try_cast(regexp_extract(salario_anual, '-\\\\s*\\\\$?(\\\\d+)', 1) as double)"), lit(0.0)) * 1000) \
    .withColumn("salario_anual_usd_estimado", 
        when(col("salario_max_usd") > 0, (col("salario_min_usd") + col("salario_max_usd")) / 2)
        .otherwise(col("salario_min_usd"))
    ) \
    .withColumn("salario_anual_brl", round(col("salario_anual_usd_estimado") * TAXA_USD_BRL, 2)) \
    .withColumn("limite_credito_limpo", regexp_replace(col("limite_credito"), r'[^\d]', '')) \
    .withColumn("limite_credito_brl", round((coalesce(expr("try_cast(limite_credito_limpo as double)"), lit(0.0)) / 100) * TAXA_USD_BRL, 2)) \
    .withColumn("valor_tx_limpo", regexp_replace(col("valor_transacoes_12m"), r'[^\d]', '')) \
    .withColumn("valor_transacoes_12m_brl", round((coalesce(expr("try_cast(valor_tx_limpo as double)"), lit(0.0)) / 100) * TAXA_USD_BRL, 2)) \
    .withColumn("qtd_tx_limpo", regexp_replace(col("qtd_transacoes_12m"), r'[^\d]', '')) \
    .withColumn("qtd_transacoes_12m", coalesce(expr("try_cast(qtd_tx_limpo as int)"), lit(0))) \
    .withColumn("data_processamento", current_timestamp()) \
    .drop("salario_min_usd", "salario_max_usd", "salario_anual_usd_estimado", "limite_credito_limpo", "valor_tx_limpo", "qtd_tx_limpo")

df_silver_credito.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver_credito_filtrado")

print("Camada Silver concluída com sucesso com os nomes das colunas mapeados!")

In [0]:
from pyspark.sql.functions import col, round, when, expr, coalesce, lit, format_number

df_silver_dados = spark.read.table("silver_credito_filtrado")

df_gold_analise = df_silver_dados \
    .withColumn(
        "salario_mensal_brl", 
        round(col("salario_anual_brl") / 12, 2).cast("decimal(18,2)")
    ) \
    .withColumn(
        "ticket_medio_transacao_brl", 
        when(col("qtd_transacoes_12m") > 0, round(col("valor_transacoes_12m_brl") / col("qtd_transacoes_12m"), 2)).otherwise(0.0).cast("decimal(18,2)")
    ) \
    .withColumn(
        "indice_uso_limite_percentual", 
        coalesce(round(expr("try_divide(valor_transacoes_12m_brl, limite_credito_brl)") * 100, 2), lit(0.0)).cast("decimal(18,2)")
    ) \
    .withColumn("salario_anual_brl", col("salario_anual_brl").cast("decimal(18,2)")) \
    .withColumn("limite_credito_brl", col("limite_credito_brl").cast("decimal(18,2)")) \
    .withColumn("valor_transacoes_12m_brl", col("valor_transacoes_12m_brl").cast("decimal(18,2)"))

df_gold_analise.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_score_credito")

spark.sql("OPTIMIZE gold_score_credito")
df_gold_analise = spark.read.table("gold_score_credito")
display(df_gold_analise)
